In [1]:
from pathlib import Path

import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI

c:\Users\NON_Academy_01\Desktop\ESTP GrnAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---------------------------------------------------------
# 1. Load documents
# ---------------------------------------------------------

DOCUMENTS_DIR = Path("data")

documents = []

for path in DOCUMENTS_DIR.glob("*.md"):
    text = path.read_text(encoding="utf-8")

    documents.append({
        "source": path.name,
        "text": text,
    })

print(f"Loaded {len(documents)} documents")


Loaded 3 documents


In [3]:
# ---------------------------------------------------------
# 2. Chunk documents
# ---------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


chunks = []

for document in documents:
    document_chunks = chunk_text(document["text"])

    for i, chunk in enumerate(document_chunks):
        chunks.append({
            "source": document["source"],
            "chunk_id": i,
            "text": chunk,
        })

print(f"Created {len(chunks)} chunks")

Created 10 chunks


In [4]:
# ---------------------------------------------------------
# 3. Create embeddings
# ---------------------------------------------------------

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
)

print(f"Embedding shape: {embeddings.shape}")



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2774.16it/s]


Embedding shape: (10, 384)


In [5]:
# ---------------------------------------------------------
# 4. Create FAISS index
# ---------------------------------------------------------

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print(f"FAISS contains {index.ntotal} vectors")


FAISS contains 10 vectors


In [6]:
# ---------------------------------------------------------
# 5. Retrieve relevant chunks
# ---------------------------------------------------------

def retrieve(question, k=3):

    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, index_number in zip(scores[0], indices[0]):
        chunk = chunks[index_number]

        results.append({
            "source": chunk["source"],
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "score": float(score),
        })

    return results


In [7]:
# ---------------------------------------------------------
# 6. Connect to LM Studio
# ---------------------------------------------------------

client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio"
)

In [8]:
# ---------------------------------------------------------
# 7. Ask Qwen using retrieved context
# ---------------------------------------------------------

def answer_question(question):

    results = retrieve(question, k=3)

    context = "\n\n".join(
        f"[Source: {r['source']}]\n{r['text']}"
        for r in results
    )

    prompt = f"""
Answer the user's question using ONLY the information
contained in the context below.

If the answer cannot be found in the context, say:
"I don't know based on the provided documents."

Do not invent information.

Context:
----------------
{context}
----------------

Question:
{question}
"""

    response = client.chat.completions.create(
        model="qwen3-8b",
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0,
    )

    answer = response.choices[0].message.content

    return answer, results

In [11]:
# ---------------------------------------------------------
# 8. Interactive loop
# ---------------------------------------------------------

while True:

    question = input("\nQuestion (or 'exit'): ")

    if question.lower() == "exit":
        break

    answer, results = answer_question(question)

    print("\n--- Retrieved chunks ---")

    for result in results:
        print(
            f"\n{result['source']} "
            f"(score={result['score']:.3f})"
        )
        print(result["text"])

    print("\n--- Answer ---")
    print(answer)


--- Retrieved chunks ---

hotel_options.md (score=0.695)
s Wi-Fi and breakfast.

Central Paris Hotel
€185 per night. Located near central Paris. Breakfast is included.

Because the standard hotel limit is €180 per night, employees need manager approval before booking the Central Paris Hotel.

Berlin

Berlin Mitte Hotel
€150 per night. Located in central Berlin. Includes Wi-Fi and breakfast.

Airport Business Hotel
€125 per night. Located near Berlin Brandenburg Airport. Includes Wi-Fi but breakfast is not included.

Hotel prices are listed as examples

travel_policy.md (score=0.637)
Business Travel Policy

Employees may travel for business when the trip is approved by their manager.

Flights longer than six hours may be booked in business class. Flights of six hours or less should be booked in economy class.

Employees should normally book hotels with a nightly cost of no more than €180. Exceptions require manager approval.

Employees may extend a business trip for personal travel, bu